# 05 - Batch Scoring

**Purpose**: Generate churn predictions for all customers on a weekly basis.

**Spec Reference**: `specs/001-churn-prediction-model/spec.md` - US1, US4

## Process
1. Validate data freshness (fail if >7 days old)
2. Load latest features
3. Load registered models
4. Score all customers for each bucket
5. Classify into risk tiers
6. Detect threshold crossings (for alerts)
7. Save predictions to Lakehouse

## Schedule
- **Frequency**: Weekly (per spec)
- **Trigger**: After data refresh pipeline completes

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import uuid

# Project imports
import sys
sys.path.append('..')
from src.utils.config import get_config
from src.utils.logging import setup_logging, get_logger, ScoringLogger
from src.data.validation import validate_data_freshness, ValidationResult
from src.scoring.batch import score_customers, create_prediction_records
from src.scoring.risk_tiers import classify_risk_tier, detect_threshold_crossings, get_risk_distribution

# Setup
setup_logging()
logger = get_logger(__name__)
config = get_config()
scoring_logger = ScoringLogger()

SCORING_DATE = datetime.now()
print(f"Scoring Date: {SCORING_DATE.strftime('%Y-%m-%d %H:%M')}")
print(f"Max Data Age: {config.data.MAX_DATA_AGE_DAYS} days")

## 1. Data Freshness Validation

⚠️ **CRITICAL**: Scoring will FAIL if data is older than 7 days.

In [ ]:
# Load features from Lakehouse
# features_df = spark.read.format("delta").load("Tables/features").toPandas()
# print(f"Loaded {len(features_df)} feature records")

In [ ]:
# Validate data freshness
# freshness_result = validate_data_freshness(
#     features_df, 
#     timestamp_column='created_at',
#     max_age_days=config.data.MAX_DATA_AGE_DAYS
# )

# if not freshness_result.passed:
#     raise ValueError(f"DATA FRESHNESS CHECK FAILED: {freshness_result.message}")
# else:
#     print(f"✓ Data freshness OK: {freshness_result.message}")

## 2. Load Trained Models

In [ ]:
# Load models from local storage or MLflow
# import joblib

# models = {}
# scalers = {}
# feature_names = {}

# for bucket in config.churn_buckets.BUCKET_NAMES:
#     models[bucket] = joblib.load(f'../models/{bucket}_model.joblib')
#     scalers[bucket] = joblib.load(f'../models/{bucket}_scaler.joblib')
#     # Load feature names from training
#     feature_names[bucket] = joblib.load(f'../models/{bucket}_features.joblib')
#     print(f"Loaded {bucket} model")

## 3. Load Previous Predictions (for alerts)

In [ ]:
# Load last week's predictions for comparison
# try:
#     previous_predictions = spark.read.format("delta").load("Tables/predictions").toPandas()
#     # Filter to most recent prediction date
#     latest_date = previous_predictions['prediction_date'].max()
#     previous_predictions = previous_predictions[previous_predictions['prediction_date'] == latest_date]
#     print(f"Loaded {len(previous_predictions)} previous predictions from {latest_date}")
# except:
#     previous_predictions = None
#     print("No previous predictions found (first run)")

## 4. Generate Predictions for All Buckets

In [ ]:
# Score all customers
# all_predictions = []
# model_version = f"v1_{SCORING_DATE.strftime('%Y%m%d')}"

# for bucket in config.churn_buckets.BUCKET_NAMES:
#     print(f"\nScoring {bucket}...")
#     
#     predictions = score_customers(
#         model=models[bucket],
#         scaler=scalers[bucket],
#         features_df=features_df,
#         feature_names=feature_names[bucket]
#     )
#     
#     predictions['churn_bucket'] = bucket
#     predictions['model_version'] = model_version
#     
#     # Add risk tier
#     predictions['risk_tier'] = predictions['churn_probability'].apply(classify_risk_tier)
#     
#     all_predictions.append(predictions)
#     
#     # Log distribution
#     dist = predictions['risk_tier'].value_counts().to_dict()
#     scoring_logger.log_risk_distribution(dist)

# predictions_df = pd.concat(all_predictions, ignore_index=True)
# print(f"\nTotal predictions: {len(predictions_df)}")

## 5. Detect Threshold Crossings (Alerts)

In [ ]:
# Detect customers who crossed into High/Critical risk
# if previous_predictions is not None:
#     # Merge current and previous
#     merged = predictions_df.merge(
#         previous_predictions[['customer_id', 'churn_bucket', 'churn_probability']],
#         on=['customer_id', 'churn_bucket'],
#         suffixes=('', '_prev'),
#         how='left'
#     )
#     
#     # Detect crossings
#     merged['alert_triggered'] = detect_threshold_crossings(
#         merged['churn_probability'],
#         merged['churn_probability_prev'],
#         threshold=config.risk_tiers.HIGH_THRESHOLD
#     )
#     
#     merged['probability_change'] = merged['churn_probability'] - merged['churn_probability_prev']
#     
#     predictions_df = merged
#     
#     alert_count = predictions_df['alert_triggered'].sum()
#     print(f"\n⚠️ ALERTS TRIGGERED: {alert_count} customers crossed into High risk")
# else:
#     predictions_df['alert_triggered'] = False
#     predictions_df['probability_change'] = None
#     predictions_df['churn_probability_prev'] = None

## 6. Create Output Records

In [ ]:
# Format predictions per contract schema
# predictions_output = predictions_df.copy()

# # Add required columns
# predictions_output['prediction_id'] = [str(uuid.uuid4()) for _ in range(len(predictions_output))]
# predictions_output['prediction_date'] = SCORING_DATE.date()
# predictions_output['created_at'] = datetime.now()

# # Rename columns to match schema
# predictions_output = predictions_output.rename(columns={
#     'churn_probability_prev': 'previous_probability'
# })

# # Select final columns
# output_columns = [
#     'prediction_id', 'customer_id', 'prediction_date', 'model_version',
#     'churn_bucket', 'churn_probability', 'risk_tier',
#     'previous_probability', 'probability_change', 'alert_triggered', 'created_at'
# ]
# predictions_output = predictions_output[output_columns]

# print(f"Prepared {len(predictions_output)} prediction records")
# predictions_output.head()

## 7. Save to Lakehouse

In [ ]:
# Save predictions to Delta table
# spark_df = spark.createDataFrame(predictions_output)
# spark_df.write.format("delta").mode("append").save("Tables/predictions")
# print("✓ Predictions saved to Lakehouse!")

## 8. Summary

In [ ]:
# Scoring summary
# print("\n" + "="*50)
# print("BATCH SCORING SUMMARY")
# print("="*50)
# print(f"Scoring Date: {SCORING_DATE.strftime('%Y-%m-%d %H:%M')}")
# print(f"Customers Scored: {predictions_output['customer_id'].nunique()}")
# print(f"Total Predictions: {len(predictions_output)}")
# print(f"Alerts Triggered: {predictions_output['alert_triggered'].sum()}")

# print("\nRisk Distribution by Bucket:")
# for bucket in config.churn_buckets.BUCKET_NAMES:
#     bucket_preds = predictions_output[predictions_output['churn_bucket'] == bucket]
#     dist = bucket_preds['risk_tier'].value_counts()
#     print(f"\n  {bucket}:")
#     for tier, count in dist.items():
#         print(f"    {tier}: {count}")

## Next Steps

1. **Dashboard Refresh**: Power BI will auto-refresh via DirectLake
2. **Alerts**: If alerts > 0, Power Automate will send notifications
3. **Review**: Check high-risk customers in dashboard